# Business Analysis Notebook

This notebook contains the data checks, KPI calculations, and analysis for the **Automotive Business Growth and Sales Analytics** project.

> **Important:** all dataset values and results in this notebook are synthetic. They are not historical Asta Innovations results.

## Workflow
1. Load the raw CSV data.
2. Validate structure and relationships.
3. Calculate core KPIs.
4. Answer business questions.
5. Produce reproducible tables for reporting/Power BI.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == 'python':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'

customers = pd.read_csv(RAW / 'customers.csv')
products = pd.read_csv(RAW / 'products.csv')
sales_reps = pd.read_csv(RAW / 'sales_reps.csv')
opportunities = pd.read_csv(RAW / 'opportunities.csv')
sales = pd.read_csv(RAW / 'sales.csv')
market_analysis = pd.read_csv(RAW / 'market_analysis.csv')

print('Loaded:')
for name, df in {
    'customers': customers,
    'products': products,
    'sales_reps': sales_reps,
    'opportunities': opportunities,
    'sales': sales,
    'market_analysis': market_analysis,
}.items():
    print(f'- {name}: {len(df):,} rows x {len(df.columns)} columns')


Loaded:
- customers: 500 rows x 8 columns
- products: 15 rows x 4 columns
- sales_reps: 20 rows x 4 columns
- opportunities: 2,000 rows x 9 columns
- sales: 1,000 rows x 10 columns
- market_analysis: 5 rows x 4 columns


## 1. Data-quality validation

Before analysing the data, I checked the table structure, keys, values, and relationships.

In [2]:
# Required tables and primary-key uniqueness
required = {
    'customers': ('customer_id', customers),
    'products': ('product_id', products),
    'sales_reps': ('sales_rep_id', sales_reps),
    'opportunities': ('opportunity_id', opportunities),
    'sales': ('sale_id', sales),
}

for name, (key, df) in required.items():
    assert df[key].notna().all(), f'{name}: null primary key'
    assert df[key].is_unique, f'{name}: duplicate primary key'
    print(f'{name}: primary-key check passed')

for name, (_, df) in required.items():
    assert df.isna().sum().sum() == 0, f'{name}: missing values'
print('Core-table missing-value checks passed')

assert (sales['quantity'] > 0).all()
assert (sales['unit_price'] >= 0).all()
assert (sales['revenue'] >= 0).all()
assert (opportunities['expected_value'] >= 0).all()
assert np.isclose(sales['revenue'], sales['quantity'] * sales['unit_price']).all()
print('Numeric and revenue-arithmetic checks passed')


customers: primary-key check passed
products: primary-key check passed
sales_reps: primary-key check passed
opportunities: primary-key check passed
sales: primary-key check passed
Core-table missing-value checks passed
Numeric and revenue-arithmetic checks passed


In [3]:
# Referential integrity
assert set(opportunities['customer_id']).issubset(set(customers['customer_id']))
assert set(opportunities['product_id']).issubset(set(products['product_id']))
assert set(opportunities['sales_rep_id']).issubset(set(sales_reps['sales_rep_id']))
assert set(sales['customer_id']).issubset(set(customers['customer_id']))
assert set(sales['product_id']).issubset(set(products['product_id']))
assert set(sales['sales_rep_id']).issubset(set(sales_reps['sales_rep_id']))
assert set(sales['opportunity_id']).issubset(set(opportunities['opportunity_id']))
print('Referential-integrity checks passed')


Referential-integrity checks passed


## 2. Core KPIs

These KPIs describe the synthetic dataset only.

In [4]:
total_revenue = sales['revenue'].sum()
total_sales = len(sales)
total_opportunities = len(opportunities)
won_opportunities = (opportunities['status'] == 'Won').sum()
conversion_rate = won_opportunities / total_opportunities
open_pipeline = opportunities.loc[opportunities['status'] == 'Open', 'expected_value'].sum()
avg_sale_value = total_revenue / total_sales

kpis = pd.Series({
    'Total Revenue': total_revenue,
    'Total Sales': total_sales,
    'Total Opportunities': total_opportunities,
    'Won Opportunities': won_opportunities,
    'Conversion Rate': conversion_rate,
    'Open Pipeline Value': open_pipeline,
    'Average Sale Value': avg_sale_value,
})
kpis


Total Revenue          2.839260e+09
Total Sales            1.000000e+03
Total Opportunities    2.000000e+03
Won Opportunities      4.440000e+02
Conversion Rate        2.220000e-01
Open Pipeline Value    2.797136e+09
Average Sale Value     2.839260e+06
dtype: float64

In [5]:
# Customer metrics used by the Power BI report
customer_sales = sales.groupby("customer_id").size()
total_customers = sales["customer_id"].nunique()
repeat_customers = (customer_sales > 1).sum()
average_customer_value = total_revenue / total_customers

customer_metrics = pd.Series({
    "Customers with Sales": total_customers,
    "Repeat Customers": repeat_customers,
    "Average Customer Value": average_customer_value,
})
customer_metrics


Customers with Sales      2.710000e+02
Repeat Customers          2.180000e+02
Average Customer Value    1.047697e+07
dtype: float64

## 3. Business questions

In [6]:
# Revenue by region
revenue_by_region = (
    sales.groupby('region', as_index=False)['revenue']
    .sum()
    .sort_values('revenue', ascending=False)
)
revenue_by_region['revenue_share_pct'] = revenue_by_region['revenue'] / total_revenue * 100
revenue_by_region


,region,revenue,revenue_share_pct
3,South,776035000.0,27.332298
2,North,667520000.0,23.510351
4,West,524320000.0,18.466784
1,East,463150000.0,16.312349
0,Central,408235000.0,14.378218


In [7]:
# Revenue by product
revenue_by_product = (
    sales.merge(products[['product_id', 'product_name', 'category']], on='product_id', how='left')
    .groupby(['product_name', 'category'], as_index=False)['revenue']
    .sum()
    .sort_values('revenue', ascending=False)
)
revenue_by_product.head(10)


,product_name,category,revenue
3,EcoHaul Pro,Electric Vehicle,495900000.0
10,HaulMaster 30,Heavy Truck,492750000.0
9,HaulMaster 20,Heavy Truck,451350000.0
8,FleetPro X2,Commercial Vehicle,352500000.0
7,FleetPro X1,Commercial Vehicle,231250000.0
2,EcoHaul,Electric Vehicle,206400000.0
13,UrbanMover,Utility Vehicle,168750000.0
0,CargoMax 10,Light Truck,163850000.0
14,UrbanMover Plus,Utility Vehicle,125550000.0
1,CargoMax 15,Light Truck,80500000.0


In [8]:
# Conversion by lead source
lead_source_performance = (
    opportunities.groupby('lead_source')
    .agg(
        opportunities=('opportunity_id', 'count'),
        won_opportunities=('status', lambda s: (s == 'Won').sum()),
        pipeline_value=('expected_value', lambda s: s[opportunities.loc[s.index, 'status'].eq('Open')].sum())
    )
    .reset_index()
)
lead_source_performance['conversion_rate_pct'] = (
    lead_source_performance['won_opportunities'] / lead_source_performance['opportunities'] * 100
)
lead_source_performance.sort_values('conversion_rate_pct', ascending=False)


,lead_source,opportunities,won_opportunities,pipeline_value,conversion_rate_pct
2,Partner,322,82,435147276.0,25.465839
0,Direct Outreach,359,88,510839896.0,24.512535
4,Trade Show,254,58,356791218.0,22.834646
3,Referral,399,82,567912388.0,20.551378
5,Website,411,84,599488511.0,20.437956
1,Email,255,50,326956469.0,19.607843


In [9]:
# Monthly revenue trend
sales['sale_date'] = pd.to_datetime(sales['sale_date'])
monthly_revenue = (
    sales.assign(month=sales['sale_date'].dt.to_period('M').astype(str))
    .groupby('month', as_index=False)['revenue']
    .sum()
)
monthly_revenue


,month,revenue
0,2025-01,2850000.0
1,2025-02,33215000.0
2,2025-03,62255000.0
3,2025-04,128695000.0
4,2025-05,118210000.0
5,2025-06,192690000.0
6,2025-07,143565000.0
7,2025-08,172445000.0
8,2025-09,149350000.0
9,2025-10,179475000.0


## 4. Reproducibility checks

The final checks compare the main KPI results with the expected values documented for this synthetic dataset.

In [10]:
assert total_revenue == 2_839_260_000
assert total_sales == 1_000
assert total_opportunities == 2_000
assert won_opportunities == 444
assert round(conversion_rate * 100, 2) == 22.20
assert open_pipeline == 2_797_135_758
print('Headline KPI reproducibility checks passed')

assert total_customers == 271
assert repeat_customers == 218
assert round(average_customer_value, 2) == 10_476_974.17


Headline KPI reproducibility checks passed


## Interpretation boundary

The tables above are analytical outputs from the synthetic dataset. They must not be presented as historical Asta Innovations revenue, conversion, pipeline, product performance, or growth results.
